# 🎬 CogVideoX-5B I2V on Kaggle P100

在 Kaggle P100（16GB）上跑 CogVideoX-5B 图片生视频。

## 使用前设置
- **Settings → Accelerator → GPU P100**（16GB）
- **Settings → Internet → On**

## 图片路径
你的图片已上传到 Kaggle Dataset，路径见下方 IMAGE_PATH。


## 1. 安装依赖 & 登录 HuggingFace

CogVideoX 需要 HuggingFace token。

1. 打开 https://huggingface.co/settings/tokens → 创建 token
2. 打开 https://huggingface.co/zai-org/CogVideoX-5b-I2V → 点 **Agree and access repository**
3. 把 token 填到下面代码的 `HF_TOKEN` 变量里


In [ ]:
import os, subprocess, time, json, sys
from pathlib import Path

# ===== 填你的 HuggingFace Token =====
HF_TOKEN="hf_trj...jKJC"  # https://huggingface.co/settings/tokens 获取
# ====================================

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('HuggingFace logged in')
else:
    print('Warning: HF_TOKEN is empty! Fill it in above.')

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Install diffusers + deps
!pip install -q diffusers>=0.32.0 transformers accelerate torchvision pillow opencv-python imageio[ffmpeg]

import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'  GPU: {props.name}, {props.total_memory/1024**3:.1f}GB VRAM')
    print(f'  Compute Capability: {props.major}.{props.minor}')
print('Done')

## 2. 配置输入

In [ ]:
# ========== 修改这里 ==========

# 1) 你的图片路径（Kaggle Dataset）
IMAGE_PATH = "/kaggle/input/datasets/flashlee/temp-iamge-01/character_template.png"

# 2) 提示词
PROMPT = "一个可爱的卡通妈妈在给宝宝洗头发，温柔的水流，温馨动画风格，柔和色彩，缓慢摆动"
NEGATIVE_PROMPT = "ugly, blurry, deformed, text, watermark, complex background, realistic, distorted face"

# 3) 视频参数（P100 友好）
NUM_FRAMES = 9        # 少帧数省显存
NUM_STEPS = 20         # 少步数省时间
GUIDANCE_SCALE = 6.0   # CFG强度
SEED = -1              # -1=随机

# 4) 分辨率（P100 16GB 极限）
TARGET_W = 320
TARGET_H = 240

# ========== 不要改下面 ==========
import random, torch, os
if SEED < 0: SEED = random.randint(0, 2**31-1)
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

if not os.path.exists(IMAGE_PATH):
    print(f'File not found: {IMAGE_PATH}')
    print('Try: find /kaggle/input -name "*.png"')
    !find /kaggle/input -name "*.png"
    raise FileNotFoundError(IMAGE_PATH)

print(f'Image: {Path(IMAGE_PATH).name}')
print(f'Prompt: {PROMPT[:60]}...')
print(f'{NUM_FRAMES} frames @ {NUM_STEPS} steps, seed={SEED}')
print(f'Target: {TARGET_W}x{TARGET_H}')

## 3. 加载 CogVideoX-5B

首次下载 ~10GB。

In [ ]:
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image

MODEL_ID = 'zai-org/CogVideoX-5b-I2V'

# === 安装 xformers：替代 SDPA，兼容 P100 (sm60) ===
# CogVideoX 的 3D 注意力掩码在 P100 上不被 Mem-Efficient SDPA 支持
# xformers 的 memory_efficient_attention 在 sm60+ 兼容性更好
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xformers'])

from diffusers.utils.import_utils import is_xformers_available
print(f'xformers available: {is_xformers_available()}')

print(f'Loading {MODEL_ID}...')
t0 = time.time()

pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

# === 激进的内存优化 ===
pipe.enable_xformers_memory_efficient_attention()  # xformers 替代 SDPA
pipe.enable_attention_slicing(slice_size=1)  # 最激进：一次只处理 1 个 token
pipe.vae.enable_slicing()                    # VAE 分片
pipe.vae.enable_tiling()                     # VAE 分块（省显存）
pipe.enable_sequential_cpu_offload()         # 单层 offload（省显存）
print('Memory optimizations enabled')

n_gpus = torch.cuda.device_count()
print(f'Model loaded in {time.time()-t0:.0f}s')
for i in range(n_gpus):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated(i) / 1024**3
        free = torch.cuda.mem_get_info(i)[0] / 1024**3
        print(f'  GPU {i}: {alloc:.1f}GB allocated, {free:.1f}GB free')

## 4. 运行推理

In [ ]:
print(f'Loading image: {IMAGE_PATH}')
image = Image.open(IMAGE_PATH).convert('RGB')
print(f'Original size: {image.size}')

# Resize + center pad to target
W, H = image.size
scale = min(TARGET_W/W, TARGET_H/H)
new_w, new_h = int(W*scale), int(H*scale)
image = image.resize((new_w, new_h), Image.LANCZOS)

from PIL import ImageOps
pad_w = TARGET_W - new_w
pad_h = TARGET_H - new_h
padding = (pad_w//2, pad_h//2, pad_w-pad_w//2, pad_h-pad_h//2)
image = ImageOps.expand(image, padding, fill=(255,255,255))
print(f'Resized to: {image.size}')

print(f'Generating {NUM_FRAMES} frames ({NUM_FRAMES/8:.1f}s @ 8fps)...')
t0 = time.time()

generator = torch.Generator(device='cpu').manual_seed(SEED)

with torch.inference_mode():
    video_frames = pipe(
        prompt=PROMPT,
        image=image,
        num_frames=NUM_FRAMES,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_STEPS,
        generator=generator,
        negative_prompt=NEGATIVE_PROMPT,
    ).frames[0]

elapsed = time.time() - t0
print(f'Done in {elapsed:.0f}s ({elapsed/NUM_FRAMES:.1f}s/frame)')

## 5. 保存 & 预览

In [ ]:
output_file = str(OUTPUT_DIR / 'cogvideo_output.mp4')
export_to_video(video_frames, output_file, fps=8)

file_size = os.path.getsize(output_file) / 1024**2
print(f'Saved: {output_file}')
print(f'  Size: {file_size:.1f} MB')
print(f'  Duration: {NUM_FRAMES/8:.1f}s @ 8fps')

from IPython.display import Video, display
display(Video(output_file, width=360, embed=True))

## 6. (可选) 换提示词再跑一次

In [ ]:
PROMPT_ALT = "a cute cartoon baby crawling on a soft mat, warm sunlight, gentle motion, pastel colors"

generator = torch.Generator(device='cpu').manual_seed(random.randint(0, 2**31-1))
with torch.inference_mode():
    frames2 = pipe(
        prompt=PROMPT_ALT,
        image=image,
        num_frames=NUM_FRAMES,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_STEPS,
        generator=generator,
        negative_prompt=NEGATIVE_PROMPT,
    ).frames[0]

output2 = str(OUTPUT_DIR / 'cogvideo_alt.mp4')
export_to_video(frames2, output2, fps=8)
print(f'Alternative: {output2}')
display(Video(output2, width=360, embed=True))